In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_date, datediff, lit

data = [
    ("U1", "2024-01-01"),
    ("U1", "2024-01-02"),
    ("U1", "2024-01-03"),
    ("U1", "2024-01-05"),
    ("U2", "2024-01-01"),
    ("U2", "2024-01-03"),
    ("U2", "2024-01-04")
]

columns = ["user_id", "activity_date"]

df = spark.createDataFrame(data, columns) \
          .withColumn("activity_date", to_date("activity_date"))

df = df.dropDuplicates(["user_id", "activity_date"])

window_spec = Window.partitionBy("user_id").orderBy("activity_date")

df = df.withColumn("rn", row_number().over(window_spec)) \
       .withColumn("group_key", datediff(col("activity_date"), to_date(lit("1970-01-01"))) - col("rn"))

result_df = df.groupBy("user_id", "group_key") \
              .count() \
              .filter(col("count") >= 3)

result_df.show()